In [2]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.4 MB/s eta 0:00:00


In [5]:
# -*- coding: utf-8 -*-
"""
INFORME AUTOMÁTICO – Modelo predictivo + gráficas documentadas en PDF

Qué hace:
1) Carga y limpia "hotel_dataset.csv".
2) Calcula métricas derivadas (RevPAR, ADR) y variables de calendario.
3) Entrena una REGRESIÓN LINEAL (X = Habitaciones_Totales, Habitaciones_Ocupadas).
4) Evalúa (RMSE, MAE, R²) y genera gráficas:
   - Paridad (Predicho vs Real) con línea ideal
   - Heatmap de correlación
   - Tendencia mensual de Ingresos
   - Distribución (histograma) de Ingresos
   - Distribución de Ingresos por día de la semana (boxplot)
   - (Opcional) Histograma de RevPAR si se pudo calcular
5) Compone un PDF con: descripción, métricas, cada gráfica + CONCLUSIÓN
   y un bloque de Conclusiones Generales.

Requisitos: pandas, numpy, matplotlib, seaborn, reportlab, scikit-learn
"""

import io
import math
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ReportLab (PDF)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch


# =========================================================
# Utilidades generales
# =========================================================
def fig_to_buf(fig, dpi=130):
    """Convierte una figura de matplotlib en buffer PNG para insertar en PDF."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=dpi)
    buf.seek(0)
    plt.close(fig)
    return buf


def cargar_y_preparar(ruta="hotel_dataset.csv"):
    """Carga datos, fuerza tipos, crea métricas derivadas y calendario."""
    df = pd.read_csv(ruta)
    df["Fecha"] = pd.to_datetime(df.get("Fecha"), errors="coerce")

    for c in ["Ingresos_Habitaciones", "Habitaciones_Totales", "Habitaciones_Ocupadas"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Métricas derivadas (para análisis, no al modelo)
    df["RevPAR"] = np.where((df.get("Habitaciones_Totales", np.nan) > 0),
                            df["Ingresos_Habitaciones"] / df["Habitaciones_Totales"], np.nan)
    df["ADR"] = np.where((df.get("Habitaciones_Ocupadas", np.nan) > 0),
                         df["Ingresos_Habitaciones"] / df["Habitaciones_Ocupadas"], np.nan)

    # Calendario
    if "Fecha" in df.columns:
        df["Mes"] = df["Fecha"].dt.month
        df["Dia_Semana"] = df["Fecha"].dt.day_name()
    else:
        df["Mes"] = np.nan
        df["Dia_Semana"] = np.nan

    return df


def entrenar_regresion(df):
    """Entrena Regresión Lineal y devuelve modelo, split y métricas."""
    X = df[["Habitaciones_Totales", "Habitaciones_Ocupadas"]].copy()
    y = df["Ingresos_Habitaciones"].copy()

    # Imputación simple (por si hay NaN)
    X = X.fillna(X.median(numeric_only=True))
    y = y.fillna(y.median())

    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(x_tr, y_tr)
    y_pr = model.predict(x_te)

    rmse = float(np.sqrt(mean_squared_error(y_te, y_pr)))
    mae = float(mean_absolute_error(y_te, y_pr))
    r2 = float(r2_score(y_te, y_pr))

    metrics = dict(RMSE=rmse, MAE=mae, R2=r2,
                   Intercepto=float(model.intercept_),
                   Coeficientes=list(map(float, model.coef_)))
    split = dict(x_train=x_tr, x_test=x_te, y_train=y_tr, y_test=y_te, y_pred=y_pr)

    return model, split, metrics


# =========================================================
# Gráficas (cada una devuelve buffer y una conclusión textual)
# =========================================================
def grafica_paridad(y_test, y_pred):
    fig = plt.figure(figsize=(9, 6))
    sns.scatterplot(x=y_test, y=y_pred, color="red", label="Predicciones", s=30, alpha=0.85)
    vmin = float(min(y_test.min(), y_pred.min()))
    vmax = float(max(y_test.max(), y_pred.max()))
    plt.plot([vmin, vmax], [vmin, vmax], color="blue", linewidth=2, label="Línea ideal")
    plt.xlim(vmin, vmax); plt.ylim(vmin, vmax)
    plt.xlabel("Valores reales"); plt.ylabel("Predicciones")
    plt.title("Paridad: Predicción vs Valor Real")
    plt.grid(True, alpha=0.35); plt.legend()
    buf = fig_to_buf(fig)

    # Conclusión automática según R²
    r2 = float(r2_score(y_test, y_pred))
    if r2 >= 0.5:
        concl = f"La nube de puntos se aproxima a la diagonal; el ajuste es sólido (R²={r2:.2f})."
    elif r2 > 0:
        concl = f"Hay aproximación parcial a la diagonal; el ajuste es moderado (R²={r2:.2f})."
    else:
        concl = f"Se observa dispersión respecto a la diagonal; el ajuste es bajo (R²={r2:.2f})."
    return buf, concl


def grafica_heatmap(df):
    cand = ["Ingresos_Habitaciones", "Habitaciones_Totales", "Habitaciones_Ocupadas", "ADR", "RevPAR"]
    num_cols = [c for c in cand if c in df.columns]
    if len(num_cols) < 2:
        return None, "No hay suficientes columnas numéricas para el mapa de calor."
    corr = df[num_cols].corr()
    fig = plt.figure(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=.5)
    plt.title("Mapa de Calor: Correlación de Métricas")
    buf = fig_to_buf(fig)

    # Variable más relacionada con Ingresos
    if "Ingresos_Habitaciones" in corr.columns:
        corrs = corr["Ingresos_Habitaciones"].drop("Ingresos_Habitaciones", errors="ignore").sort_values(ascending=False)
        top_var = corrs.index[0]
        top_val = float(corrs.iloc[0])
        concl = f"La variable más relacionada con Ingresos es {top_var} (ρ={top_val:.2f})."
    else:
        concl = "Las correlaciones permiten ver relaciones lineales entre las métricas clave."
    return buf, concl


def grafica_tendencia(df, agregacion="mean"):
    if "Fecha" not in df.columns or df["Fecha"].isna().all():
        return None, "No hay fechas válidas para graficar tendencia."

    serie = getattr(df.groupby(df["Fecha"].dt.to_period("M"))["Ingresos_Habitaciones"], agregacion)().reset_index()
    serie["Fecha"] = serie["Fecha"].dt.to_timestamp()

    fig = plt.figure(figsize=(12, 5))
    plt.plot(serie["Fecha"], serie["Ingresos_Habitaciones"], marker="o", linewidth=2)
    plt.title("Tendencia de Ingresos (Promedio Mensual)" if agregacion == "mean" else "Tendencia de Ingresos (Suma Mensual)")
    plt.xlabel("Fecha"); plt.ylabel("Ingresos")
    plt.grid(True, linestyle="--", alpha=0.5)
    buf = fig_to_buf(fig)

    # Inclinación aproximada para la conclusión
    x = np.arange(len(serie))
    if len(serie) >= 2 and not np.isnan(serie["Ingresos_Habitaciones"]).all():
        slope = np.polyfit(x, serie["Ingresos_Habitaciones"], 1)[0]
        if slope > 0:
            concl = "La tendencia mensual es creciente."
        elif slope < 0:
            concl = "La tendencia mensual es decreciente."
        else:
            concl = "La tendencia mensual es estable."
    else:
        concl = "No se puede inferir una tendencia por falta de datos."
    return buf, concl


def grafica_hist_ingresos(df):
    if "Ingresos_Habitaciones" not in df.columns:
        return None, "No existen ingresos para graficar su distribución."
    ser = df["Ingresos_Habitaciones"].dropna()
    fig = plt.figure(figsize=(10, 6))
    sns.histplot(ser, bins=30, kde=True, edgecolor="black")
    plt.title("Distribución de Ingresos de Habitaciones")
    plt.xlabel("Ingresos"); plt.ylabel("Frecuencia")
    plt.grid(axis="y", alpha=0.3)
    buf = fig_to_buf(fig)

    # Asimetría
    skew = float(ser.skew()) if len(ser) > 0 else 0.0
    if skew > 0.5:
        concl = f"La distribución es asimétrica a la derecha (sesgo positivo, skew={skew:.2f})."
    elif skew < -0.5:
        concl = f"La distribución es asimétrica a la izquierda (sesgo negativo, skew={skew:.2f})."
    else:
        concl = f"La distribución es aproximadamente simétrica (skew={skew:.2f})."
    return buf, concl


def grafica_box_dia_semana(df):
    if "Mes" not in df.columns or "Ingresos_Habitaciones" not in df.columns:
        return None, "Faltan columnas para la distribución por Mes."
    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(x="Mes", y="Ingresos_Habitaciones", data=df, palette="Set3")
    plt.title("Distribución de Ingresos por Meses")
    plt.xlabel("Meses"); plt.ylabel("Ingresos")
    plt.xticks(rotation=45); plt.grid(axis="y", alpha=0.3)
    buf = fig_to_buf(fig)

    med = df.groupby("Mes")["Ingresos_Habitaciones"].median().sort_values(ascending=False)
    top_day = med.index[0]
    concl = f"El Mes con mayor mediana de ingresos es {top_day}."
    return buf, concl


def grafica_hist_revpar(df):
    if "RevPAR" not in df.columns or df["RevPAR"].dropna().empty:
        return None, "No fue posible calcular RevPAR para su distribución."
    ser = df["RevPAR"].dropna()
    fig = plt.figure(figsize=(10, 6))
    sns.histplot(ser, bins=30, kde=True, edgecolor="black", color="#6ca0dc")
    plt.title("Distribución de RevPAR")
    plt.xlabel("RevPAR"); plt.ylabel("Frecuencia")
    plt.grid(axis="y", alpha=0.3)
    buf = fig_to_buf(fig)

    skew = float(ser.skew())
    concl = (
        f"La distribución de RevPAR presenta sesgo {'positivo' if skew>0 else 'negativo' if skew<0 else 'nulo'} (skew={skew:.2f})."
    )
    return buf, concl


# =========================================================
# PDF – armado del informe
# =========================================================
def generar_pdf(df, metrics, figuras, output="Informe_Modelo_Hotel.pdf"):
    """
    Crea un PDF con secciones: resumen, métricas, cada gráfica con explicación y conclusión,
    y un cierre con conclusiones generales.
    """
    doc = SimpleDocTemplate(output, pagesize=letter)
    styles = getSampleStyleSheet()
    # estilos propios
    for name, cfg in [
        ("H1", dict(fontSize=18, leading=22, spaceAfter=12, fontName="Helvetica-Bold")),
        ("H2", dict(fontSize=14, leading=18, spaceBefore=10, spaceAfter=6, fontName="Helvetica-Bold")),
        ("P",  dict(fontSize=11, leading=14)),
    ]:
        if name not in styles:
            styles.add(ParagraphStyle(name=name, **cfg))

    elems = []
    # Portada breve
    elems.append(Paragraph("Informe – Modelo Predictivo de Ingresos Hoteleros", styles["H1"]))
    elems.append(Paragraph(f"Fecha: {datetime.now():%Y-%m-%d}", styles["P"]))
    elems.append(Spacer(0, 6))
    elems.append(Paragraph(f"Observaciones: N={len(df):,}", styles["P"]))
    elems.append(Spacer(0, 12))

    # Resumen del modelo y métricas
    elems.append(Paragraph("Modelo y Métricas", styles["H2"]))
    r2, rmse, mae = metrics["R2"], metrics["RMSE"], metrics["MAE"]
    resumen = (
        "Se entrenó una <b>Regresión Lineal</b> con variables predictoras "
        "<b>Habitaciones_Totales</b> y <b>Habitaciones_Ocupadas</b>. "
        "La evaluación se realizó con un 20% de los datos como conjunto de prueba."
        "<br/><br/>"
        f"<b>R²:</b> {r2:.4f} &nbsp; "
        f"<b>RMSE:</b> ${rmse:,.2f} &nbsp; "
        f"<b>MAE:</b> ${mae:,.2f}."
    )
    elems.append(Paragraph(resumen, styles["P"]))
    elems.append(Spacer(0, 10))

    # === Secciones por figura ===
    def add_figure(title, desc, buf, concl):
        elems.append(Paragraph(title, styles["H2"]))
        elems.append(Paragraph(desc, styles["P"]))
        if buf is not None:
            elems.append(Image(buf, width=6.4*inch, height=4.2*inch))
        elems.append(Paragraph(f"<b>Conclusión:</b> {concl}", styles["P"]))
        elems.append(Spacer(0, 12))

    # 1) Paridad
    add_figure(
        "1) Paridad Predicción vs Real",
        "Compara las predicciones del modelo con los valores reales. La línea azul representa la predicción ideal (y=x).",
        figuras["paridad"]["img"],
        figuras["paridad"]["conclusion"],
    )

    # 2) Heatmap
    add_figure(
        "2) Mapa de Calor de Correlación",
        "Mide la fuerza de relación lineal entre métricas clave (|ρ| cercano a 1 indica relación fuerte).",
        figuras["heatmap"]["img"],
        figuras["heatmap"]["conclusion"],
    )

    # 3) Tendencia
    add_figure(
        "3) Tendencia de Ingresos (Promedio Mensual)",
        "Serie mensual del promedio de ingresos, útil para detectar estacionalidad y direcciones de cambio.",
        figuras["tendencia"]["img"],
        figuras["tendencia"]["conclusion"],
    )

    # 4) Histograma de Ingresos
    add_figure(
        "4) Distribución de Ingresos",
        "Histograma con densidad estimada (KDE) para evaluar la forma de la distribución y presencia de colas.",
        figuras["hist_ingresos"]["img"],
        figuras["hist_ingresos"]["conclusion"],
    )

    # 5) Boxplot por día de la semana
    add_figure(
        "5) Ingresos por Día de la Semana",
        "Boxplots por día permiten comparar medianas y la dispersión de ingresos.",
        figuras["box_semana"]["img"],
        figuras["box_semana"]["conclusion"],
    )

    # 6) (Opcional) RevPAR
    if "hist_revpar" in figuras:
        add_figure(
            "6) Distribución de RevPAR (opcional)",
            "Muestra la variabilidad del RevPAR; útil para estrategias de pricing y segmentación.",
            figuras["hist_revpar"]["img"],
            figuras["hist_revpar"]["conclusion"],
        )

    # Conclusiones generales
    conclusiones = []
    # Del R²
    if r2 >= 0.5:
        conclusiones.append("El modelo explica una proporción relevante de la varianza; es útil para soporte de decisiones.")
    elif r2 > 0:
        conclusiones.append("El modelo muestra poder predictivo moderado; se recomienda complementarlo con más variables.")
    else:
        conclusiones.append("El modelo no supera a un predictor simple; conviene incorporar más drivers y revisar supuestos.")

    # De heatmap
    conclusiones.append(figuras["heatmap"]["conclusion"])

    # De tendencia
    conclusiones.append(figuras["tendencia"]["conclusion"])

    # De boxplot (días)
    conclusiones.append(figuras["box_semana"]["conclusion"])

    # Conclusiones, sugerencias e integrantes
    elems.append(Paragraph("Conclusiones Generales", styles["H2"]))
    elems.append(Paragraph("• " + "<br/>• ".join(conclusiones), styles["P"]))
    elems.append(Spacer(0, 6))

    elems.append(Paragraph(
    "Sugerencias: incorporar variables externas (eventos, competencia), probar modelos no lineales "
    "(árboles/boosting) y evaluar sobre ventanas temporales (validación temporal) para robustez.",
    styles["P"]
    ))

    # Integrantes (después de Sugerencias)
    integrantes_html = """
    <br/><b>Integrantes:</b>
    <br/>1. <b> Walter Colorado Gonzalez </b>
    <br/>2. <b> Yuleidy alexandra Barreto paez</b>
    <br/>3. <b> Victor Manuelle Artunduaga Gonzalez</b>
    <br/>4. <b> Maria Doris Moreno </b>
    """
    elems.append(Paragraph(integrantes_html, styles["P"]))
    elems.append(Spacer(0, 12))

    doc.build(elems)
    print(f"✅ Informe PDF generado: {output}")

# =========================================================
# EJECUCIÓN
# =========================================================
if __name__ == "__main__":
    # 1) Datos
    df = cargar_y_preparar("hotel_dataset.csv")
    print("Dimensiones:", df.shape)

    # 2) Modelo
    modelo, split, metrics = entrenar_regresion(df)
    y_test, y_pred = split["y_test"], split["y_pred"]
    print("Métricas:", metrics)

    # 3) Gráficas + conclusiones
    figs = {}

    img, concl = grafica_paridad(y_test, y_pred)
    figs["paridad"] = {"img": img, "conclusion": concl}

    img, concl = grafica_heatmap(df)
    figs["heatmap"] = {"img": img, "conclusion": concl}

    img, concl = grafica_tendencia(df, agregacion="mean")
    figs["tendencia"] = {"img": img, "conclusion": concl}

    img, concl = grafica_hist_ingresos(df)
    figs["hist_ingresos"] = {"img": img, "conclusion": concl}

    img, concl = grafica_box_dia_semana(df)
    figs["box_semana"] = {"img": img, "conclusion": concl}

    img, concl = grafica_hist_revpar(df)
    if img is not None:
        figs["hist_revpar"] = {"img": img, "conclusion": concl}

    # 4) PDF
    generar_pdf(df, metrics, figs, output="Informe_Modelo_Hotel.pdf")


Dimensiones: (1098, 16)
Métricas: {'RMSE': 4313786.023782612, 'MAE': 3406886.7324765683, 'R2': 0.8013539146016214, 'Intercepto': -646584.4875969738, 'Coeficientes': [915.5923169541632, 231807.42700729019]}


/tmp/ipython-input-137604661.py:210: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="Mes", y="Ingresos_Habitaciones", data=df, palette="Set3")


✅ Informe PDF generado: Informe_Modelo_Hotel.pdf
